In [2]:
import re
import sys
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
ranger_procd_root = Path("/oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Ranger_procd")
out_h5ad = ranger_procd_root / "ALS_SCXenium_concatenated_offset.h5ad"

# Only process directories starting with this prefix (set "" to process all)
folder_prefix = "ALS"

# CSV status file (same folder as Ranger_procd per your screenshot)
status_csv = ranger_procd_root.parent / "status.csv"  # /oak/.../Projects/RK/Spatial/status.csv

# --- Spatial offset settings (prevents cross-sample spatial mixing) ---
SPATIAL_OFFSET = 2_000_000.0  # large fixed offset (units match Xenium coords, usually microns)
SPATIAL_GRID_COLS = 10        # number of columns in gallery grid

# --- Diagnostics ---
VERBOSE = True
MAX_EXAMPLES = 5

# ============================================================
# HELPERS
# ============================================================
def dprint(msg: str) -> None:
    if VERBOSE:
        print(msg, flush=True)

def normalize_cell_ids(idx: pd.Index) -> pd.Index:
    s = idx.astype(str)
    s = s.str.replace(r"-\d+$", "", regex=True)        # drop trailing -1, -2, etc.
    s = s.str.replace(r"^cell[_\-]?", "", regex=True)  # drop leading cell_ / cell-
    return pd.Index(s)

def choose_id_column(df: pd.DataFrame) -> str:
    for c in ["cell_id", "barcode", "cell_barcode", "cell", "id"]:
        if c in df.columns:
            return c
    raise ValueError(
        "Could not find a cell ID column in cell metadata.\n"
        f"Available columns (first 120): {list(df.columns[:120])}"
    )

def find_xenium_outs(sample_dir: Path) -> Path | None:
    """
    Handles both:
      - <sample_dir>/outs
      - <sample_dir>/<sample_dir.name>/outs
    Plus a conservative fallback search.
    """
    direct = sample_dir / "outs"
    if (direct / "cell_feature_matrix.h5").exists():
        return direct

    nested = sample_dir / sample_dir.name / "outs"
    if (nested / "cell_feature_matrix.h5").exists():
        return nested

    candidates = []
    for p in sample_dir.rglob("cell_feature_matrix.h5"):
        try:
            rel = p.relative_to(sample_dir)
            if len(rel.parts) <= 6 and p.parent.name == "outs":
                candidates.append(p.parent)
        except Exception:
            continue

    if candidates:
        candidates = sorted(candidates, key=lambda x: (len(x.relative_to(sample_dir).parts), str(x)))
        return candidates[0]

    return None

def extract_sd_code_anywhere(x: str | Path) -> str | None:
    """
    Return SDxxxxx (five digits) if found, robust to:
      - SD04219
      - SD04219_BI
      - SD016_20_BI  (=> SD01620)
      - SD02022__... (=> SD02022)
      - SD042/19     (=> SD04219)
    """
    s = str(x)

    m = re.search(r"(SD\d{5})", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    m = re.search(r"(SD\d{3})[_\-](\d{2})", s, flags=re.IGNORECASE)
    if m:
        return f"{m.group(1).upper()}{m.group(2)}"

    m = re.search(r"(SD)\s*0*(\d+)\s*/\s*(\d+)", s, flags=re.IGNORECASE)
    if m:
        left = int(m.group(2))
        right = int(m.group(3))
        return f"{m.group(1).upper()}{left:03d}{right:02d}"

    return None

def extract_sample_label(sample_dir: Path, outs_dir: Path) -> str:
    """
    What goes into obs['sample'] (human-meaningful sample key).
    Preference:
      1) SDxxxxx_<SUFFIX> if present (e.g., SD04219_BI)
      2) SDxxxxx if present (e.g., SD02022)
      3) sample_dir.name as fallback (e.g., ALS5_Region_2__...)
    """
    s = str(outs_dir)

    m = re.search(r"(SD\d{5}_[A-Za-z0-9]+)", s)
    if m:
        return m.group(1)

    m = re.search(r"(SD\d{3})[_\-](\d{2})[_\-]([A-Za-z0-9]+)", s)
    if m:
        return f"{m.group(1)}{m.group(2)}_{m.group(3)}"

    sd = extract_sd_code_anywhere(s)
    if sd:
        return sd

    return sample_dir.name

def parse_status_mapping_from_csv(csv_path: Path) -> dict:
    if not csv_path.exists():
        dprint(f"WARNING: status CSV not found: {csv_path}. obs['status'] will be 'unknown'.")
        return {}

    df = pd.read_csv(csv_path)

    cols_lower = {c: str(c).strip().lower() for c in df.columns}
    status_col = next((c for c, lc in cols_lower.items() if lc in {"status", "group", "genotype"}), None)
    sample_col = next((c for c, lc in cols_lower.items() if lc in {"sample", "sample_id", "id", "case number", "case_number"}), None)

    mapping: dict[str, str] = {}

    if status_col is not None and sample_col is not None:
        dprint(f"[status.csv] Using explicit columns: sample_col='{sample_col}', status_col='{status_col}'")
        for _, row in df.iterrows():
            sd = extract_sd_code_anywhere(row.get(sample_col))
            if not sd:
                continue
            st_raw = str(row.get(status_col)).strip().lower()
            if not st_raw or st_raw == "nan":
                continue
            if "control" in st_raw:
                mapping[sd] = "control"
            elif "c9" in st_raw:
                mapping[sd] = "c9"
            elif "sporadic" in st_raw:
                mapping[sd] = "sporadic"
            else:
                mapping[sd] = st_raw
        return mapping

    section_map = {
        "SPORADIC": "sporadic",
        "C9ORF": "c9",
        "CONTROL": "control",
        "CONTROLS": "control",
    }

    first_col = df.columns[0]
    current = None
    dprint(f"[status.csv] Using section-header parsing from first column '{first_col}'")

    for _, row in df.iterrows():
        v = row.get(first_col)
        if pd.isna(v):
            continue

        txt = str(v).strip()
        if not txt:
            continue

        txt_upper = txt.upper()

        if txt_upper in {"CASE NUMBER", "CASENUMBER"}:
            continue

        if txt_upper in section_map:
            current = section_map[txt_upper]
            dprint(f"[status.csv] Enter section: {txt_upper} -> '{current}'")
            continue

        sd = extract_sd_code_anywhere(txt_upper)
        if sd and current is not None:
            mapping[sd] = current

    return mapping

def _minmax2(xy: np.ndarray) -> tuple[tuple[float, float], tuple[float, float]]:
    return (float(np.nanmin(xy[:, 0])), float(np.nanmax(xy[:, 0]))), (float(np.nanmin(xy[:, 1])), float(np.nanmax(xy[:, 1])))

def load_sample_from_outs(outs_dir: Path) -> sc.AnnData:
    h5_path = outs_dir / "cell_feature_matrix.h5"
    cells_parquet = outs_dir / "cells.parquet"
    cells_csv_gz = outs_dir / "cells.csv.gz"

    if not h5_path.exists():
        raise FileNotFoundError(f"Missing expression H5: {h5_path}")

    if not (cells_parquet.exists() or cells_csv_gz.exists()):
        raise FileNotFoundError(
            "Missing cell metadata. Expected one of:\n"
            f"- {cells_parquet}\n"
            f"- {cells_csv_gz}"
        )

    dprint(f"  [load] reading 10x h5: {h5_path}")
    adata = sc.read_10x_h5(str(h5_path), gex_only=False)
    adata.var_names_make_unique()

    dprint(f"  [load] raw matrix: n_obs={adata.n_obs}, n_vars={adata.n_vars}")
    dprint(f"  [load] obs_names (raw) examples: {list(adata.obs_names[:MAX_EXAMPLES])}")

    adata.obs_names = normalize_cell_ids(pd.Index(adata.obs_names))
    dprint(f"  [load] obs_names (normalized) examples: {list(adata.obs_names[:MAX_EXAMPLES])}")

    cells = None
    load_source = None

    if cells_parquet.exists():
        try:
            dprint(f"  [load] reading metadata parquet: {cells_parquet}")
            cells = pd.read_parquet(cells_parquet)
            load_source = "cells.parquet"
        except Exception as e:
            dprint(f"  [load] WARNING: failed parquet ({type(e).__name__}: {e}); will try CSV.gz")

    if cells is None:
        dprint(f"  [load] reading metadata csv.gz: {cells_csv_gz}")
        cells = pd.read_csv(cells_csv_gz, compression="gzip", low_memory=False)
        load_source = "cells.csv.gz"

    dprint(f"  [meta] metadata shape: {cells.shape}, columns: {len(cells.columns)}")
    id_col = choose_id_column(cells)
    dprint(f"  [meta] chosen ID column: '{id_col}' (source={load_source})")

    cells[id_col] = cells[id_col].astype(str)
    cells[id_col] = normalize_cell_ids(pd.Index(cells[id_col])).astype(str)
    cells = cells.set_index(id_col)

    if not cells.index.is_unique:
        dup_n = int(cells.index.duplicated().sum())
        examples = list(cells.index[cells.index.duplicated()].unique()[:MAX_EXAMPLES])
        raise ValueError(f"cells metadata index not unique after normalization: {dup_n} duplicates; examples={examples}")

    common_mask = adata.obs_names.isin(cells.index)
    n_common = int(common_mask.sum())
    dprint(f"  [join] overlap: {n_common}/{adata.n_obs} expression cells have metadata rows")

    if n_common == 0:
        dprint(f"  [join] AnnData obs_names examples: {list(adata.obs_names[:MAX_EXAMPLES])}")
        dprint(f"  [join] Metadata index examples: {list(cells.index[:MAX_EXAMPLES])}")
        raise ValueError("No overlapping cell IDs between expression matrix and cell metadata.")

    common = adata.obs_names[common_mask]
    adata = adata[common].copy()
    adata.obs = adata.obs.join(cells.loc[common], how="left")

    # Spatial
    candidate_pairs = [
        ("x_centroid", "y_centroid"),
        ("centroid_x", "centroid_y"),
        ("x", "y"),
        ("x_location", "y_location"),
        ("x_center", "y_center"),
    ]
    found = False
    for xcol, ycol in candidate_pairs:
        if xcol in adata.obs.columns and ycol in adata.obs.columns:
            adata.obsm["spatial"] = adata.obs[[xcol, ycol]].to_numpy()
            adata.uns["spatial_columns"] = {"x": xcol, "y": ycol, "source": load_source}
            xy = adata.obsm["spatial"].astype(float)
            (xmin, xmax), (ymin, ymax) = _minmax2(xy)
            dprint(f"  [spatial] using columns ({xcol},{ycol}) from {load_source}; "
                   f"x=[{xmin:.2f},{xmax:.2f}], y=[{ymin:.2f},{ymax:.2f}]")
            found = True
            break

    if not found:
        dprint("  [spatial] WARNING: no recognized spatial columns found; obsm['spatial'] not set")

    adata.uns["xenium_outs_path"] = str(outs_dir)
    return adata

# ============================================================
# MAIN
# ============================================================
if not ranger_procd_root.exists():
    raise FileNotFoundError(f"Missing Ranger_procd root: {ranger_procd_root}")

status_map = parse_status_mapping_from_csv(status_csv)
dprint(f"Loaded status mappings: {len(status_map)} from {status_csv}")

sample_dirs = sorted([p for p in ranger_procd_root.iterdir() if p.is_dir()])
dprint(f"Found {len(sample_dirs)} directories under {ranger_procd_root}")

adatas = []
kept = 0
skipped = 0

for sample_dir in sample_dirs:
    if folder_prefix and not sample_dir.name.startswith(folder_prefix):
        continue

    dprint(f"\n=== SAMPLE DIR: {sample_dir.name} ===")

    outs_dir = find_xenium_outs(sample_dir)
    if outs_dir is None:
        skipped += 1
        dprint(f"[SKIP] {sample_dir.name}: could not find outs/ with cell_feature_matrix.h5")
        continue
    dprint(f"[outs] {outs_dir}")

    try:
        adata = load_sample_from_outs(outs_dir)

        sample_label = extract_sample_label(sample_dir, outs_dir)
        sd_code = extract_sd_code_anywhere(sample_label)
        status = status_map.get(sd_code, "unknown") if sd_code else "unknown"

        run_id = sample_dir.name
        dprint(f"[ids] run_id='{run_id}', sample_label='{sample_label}', sd_code='{sd_code}', status='{status}'")

        # Store original Xenium cell id before prefixing
        adata.obs["xenium_cell_id"] = adata.obs_names.astype(str)

        # Prefix obs_names for global uniqueness
        adata.obs_names = pd.Index([f"{run_id}:{cid}" for cid in adata.obs["xenium_cell_id"]])
        dprint(f"[ids] obs_names prefixed examples: {list(adata.obs_names[:MAX_EXAMPLES])}")

        # Required obs columns
        adata.obs["sample"] = sample_label
        adata.obs["status"] = status

        # Optional provenance
        adata.obs["run_id"] = run_id
        adata.obs["sd_code"] = sd_code if sd_code else ""

        adata.uns["sample_dir"] = str(sample_dir)
        adata.uns["sample_label"] = sample_label

        # ------------------------------------------------------------
        # Spatial offset diagnostics + application
        # ------------------------------------------------------------
        if "spatial" in adata.obsm:
            xy = adata.obsm["spatial"].astype(float)
            (xmin, xmax), (ymin, ymax) = _minmax2(xy)

            idx = kept  # 0-based index of successfully loaded samples so far
            gx = idx % SPATIAL_GRID_COLS
            gy = idx // SPATIAL_GRID_COLS

            dprint(f"[offset] idx={idx}, grid=({gx},{gy}), offset={SPATIAL_OFFSET:g}")
            dprint(f"[offset] spatial_orig bounds: x=[{xmin:.2f},{xmax:.2f}], y=[{ymin:.2f},{ymax:.2f}]")

            shifted = xy.copy()
            shifted[:, 0] += gx * SPATIAL_OFFSET
            shifted[:, 1] += gy * SPATIAL_OFFSET

            (sxmin, sxmax), (symin, symax) = _minmax2(shifted)
            dprint(f"[offset] spatial_shifted bounds: x=[{sxmin:.2f},{sxmax:.2f}], y=[{symin:.2f},{symax:.2f}]")

            adata.obsm["spatial_orig"] = xy
            adata.obsm["spatial"] = shifted

            adata.uns["spatial_offset"] = {
                "offset": float(SPATIAL_OFFSET),
                "grid_cols": int(SPATIAL_GRID_COLS),
                "gx": int(gx),
                "gy": int(gy),
                "index": int(idx),
            }
        else:
            dprint("[offset] NOTE: no obsm['spatial'] present; skipping offset")
            adata.uns["spatial_offset"] = None

        dprint(f"[OK] {sample_dir.name} -> n_obs={adata.n_obs}, n_vars={adata.n_vars}")
        dprint(f"     status='{status}', sample='{sample_label}'")

        adatas.append(adata)
        kept += 1

    except Exception as e:
        skipped += 1
        dprint(f"[FAIL] {sample_dir.name}: {type(e).__name__}: {e}")
        continue

if not adatas:
    raise RuntimeError("No samples were successfully loaded; nothing to concatenate.")

dprint(f"\nLoaded samples: {kept}, skipped/failed: {skipped}")
dprint("Concatenating...")

adata_all = sc.concat(
    adatas,
    axis=0,
    join="outer",
    merge="same",
    label=None,
    index_unique=None,
)

# Final safety: ensure uniqueness (should already be unique via run_id prefix)
adata_all.obs_names_make_unique()

adata_all.obs["sample"] = adata_all.obs["sample"].astype("category")
adata_all.obs["status"] = adata_all.obs["status"].astype("category")

dprint("Combined AnnData:")
dprint(f"  n_obs={adata_all.n_obs}")
dprint(f"  n_vars={adata_all.n_vars}")
dprint("Status counts:")
dprint(str(adata_all.obs["status"].value_counts(dropna=False)))

# Diagnostics on combined spatial (if present)
if "spatial" in adata_all.obsm:
    xy_all = adata_all.obsm["spatial"].astype(float)
    (xmin, xmax), (ymin, ymax) = _minmax2(xy_all)
    dprint(f"[combined spatial] bounds: x=[{xmin:.2f},{xmax:.2f}], y=[{ymin:.2f},{ymax:.2f}]")
    # quick sanity: show a few run_ids and mean coords
    if "run_id" in adata_all.obs.columns:
        dprint("[combined spatial] per-run mean coords (first 10 runs):")
        df_means = (
            pd.DataFrame(xy_all, columns=["x", "y"], index=adata_all.obs_names)
            .join(adata_all.obs[["run_id"]])
            .groupby("run_id")[["x", "y"]].mean()
            .head(10)
        )
        dprint(str(df_means))

dprint(f"Writing: {out_h5ad}")
adata_all.write_h5ad(str(out_h5ad), compression="gzip")
dprint("Done.")


[status.csv] Using section-header parsing from first column 'TR19/24 JCK'
[status.csv] Enter section: SPORADIC -> 'sporadic'
[status.csv] Enter section: C9ORF -> 'c9'
[status.csv] Enter section: CONTROLS -> 'control'
Loaded status mappings: 20 from /oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/status.csv
Found 23 directories under /oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Ranger_procd

=== SAMPLE DIR: ALS1_SD01616_BI__largecells_reseg_20251215_064236_job49421199_1 ===
[outs] /oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Ranger_procd/ALS1_SD01616_BI__largecells_reseg_20251215_064236_job49421199_1/outs
  [load] reading 10x h5: /oak/stanford/scg/lab_mpsnyder/johnck/Projects/RK/Spatial/Ranger_procd/ALS1_SD01616_BI__largecells_reseg_20251215_064236_job49421199_1/outs/cell_feature_matrix.h5
  [load] raw matrix: n_obs=77518, n_vars=541
  [load] obs_names (raw) examples: ['aaaameme-1', 'aaaapajd-1', 'aaabjnoi-1', 'aaacojbg-1', 'aaaddkgc-1']
  [load] obs_